<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/HDOCK_Docking_Protein_DNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 Initialize Setup and Start Service { display-mode: "form" }

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE FOR PERSISTENT STORAGE
# ==============================================================================
import os
from google.colab import drive

drive.mount('/content/drive')

work_dir = '/content/drive/MyDrive/HDOCKlite'

# ==============================================================================
# 2. CLONE HDOCK SUBFOLDER & SETUP TOOLS
# ==============================================================================
if not os.path.exists(work_dir):
    !git clone --depth 1 https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git /content/temp_repo
    !cp -r /content/temp_repo/HDOCK {work_dir}
    !rm -rf /content/temp_repo
else:
    print("HDOCKlite already exists in Google Drive, skipping clone.")

%cd {work_dir}

# Set executable permissions on scripts
!chmod +x hdock createpl

In [ ]:
# @title 🚀 Upload Receptor and Ligand PDBs & Run HDOCK { display-mode: "form" }
# @markdown Click the play button, then upload your **Receptor** first and **Ligand** second.

from google.colab import files

print("1️⃣ Please upload the RECEPTOR PDB file:")
receptor_upload = files.upload()
receptor_pdb = list(receptor_upload.keys())[0]

print("\n2️⃣ Please upload the LIGAND PDB file:")
ligand_upload = files.upload()
ligand_pdb = list(ligand_upload.keys())[0]

print(f"\n✅ Ready! Receptor: {receptor_pdb} | Ligand: {ligand_pdb}")

print("\n⏳ Running HDOCK...")
!./hdock "{receptor_pdb}" "{ligand_pdb}" -out Hdock.out

print("\n⏳ Generating top 100 complex models...")
!./createpl Hdock.out top100.pdb -nmax 100 -complex -models

print("\n🎉 Done! Models saved to top100.pdb")

In [ ]:
# @title 🎯🧠 Interactive AI Scoring Tool (Auto-Score HDOCKlite Models) { display-mode: "form" }

!pip install biopython scipy requests torch_geometric -q
!apt-get update -qq
!apt-get install dssp -y -qq

import os
import warnings
from Bio import BiopythonWarning
warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', BiopythonWarning)

from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.nn import MessagePassing, global_mean_pool
from scipy.spatial import distance_matrix
import numpy as np

# --- 1. Define the Core Architecture ---
class EGNNLayer(MessagePassing):
    def __init__(self, node_dim):
        super().__init__(aggr='mean')
        self.node_mlp = nn.Sequential(nn.Linear(node_dim * 2 + 1, node_dim), nn.SiLU(), nn.Linear(node_dim, node_dim))
        self.coord_mlp = nn.Sequential(nn.Linear(node_dim, node_dim), nn.SiLU(), nn.Linear(node_dim, 1))

    def forward(self, x, pos, edge_index):
        row, col = edge_index
        radial = torch.sum((pos[row] - pos[col]) ** 2, dim=1).unsqueeze(1)
        return self.propagate(edge_index, x=x, pos=pos, radial=radial)

    def message(self, x_i, x_j, radial, pos_i, pos_j):
        m_ij = self.node_mlp(torch.cat([x_i, x_j, radial], dim=-1))
        pos_update = self.coord_mlp(m_ij) * (pos_i - pos_j)
        return m_ij, pos_update

    def aggregate(self, inputs, index, ptr=None, dim_size=None):
        m_ij, pos_update = inputs
        return super().aggregate(m_ij, index, ptr=ptr, dim_size=dim_size), super().aggregate(pos_update, index, ptr=ptr, dim_size=dim_size)

    def update(self, aggr_out, x, pos):
        agg_m, agg_pos = aggr_out
        return x + agg_m, pos + agg_pos

class ComplexScoringModel(nn.Module):
    def __init__(self, in_node_dim=39, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Linear(in_node_dim, hidden_dim)
        self.egnn1 = EGNNLayer(hidden_dim)
        self.egnn2 = EGNNLayer(hidden_dim)
        self.readout = nn.Sequential(nn.Linear(hidden_dim, 32), nn.SiLU(), nn.Linear(32, 1))

    def forward(self, data):
        x = self.embedding(data.x)
        x, pos = self.egnn1(x, data.pos, data.edge_index)
        x, pos = self.egnn2(x, pos, data.edge_index)
        graph_embedding = global_mean_pool(x, data.batch)
        return self.readout(graph_embedding)

# --- 2. Setup Biophysics Mappings ---
AA_MAP = {aa: i for i, aa in enumerate(['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'])}
NA_MAP = {na: i for i, na in enumerate(['A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC'])}
SS_MAP = {'H': 0, 'B': 1, 'E': 2, 'G': 3, 'I': 4, 'T': 5, 'S': 6, '-': 7}

# --- 3. Locate Model Weights & PDB Files in HDOCKlite Folder ---
work_dir = '/content/drive/MyDrive/HDOCKlite'
model_path = os.path.join(work_dir, 'egnn_docking_scorer.pth')

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model weights not found at: {model_path}")

pdb_files = [
    os.path.join(work_dir, f"model_{i}.pdb")
    for i in range(1, 101)
    if os.path.exists(os.path.join(work_dir, f"model_{i}.pdb"))
]

if not pdb_files:
    print(f"No model_1.pdb–model_100.pdb files found in {work_dir}. Exiting.")
else:
    print(f"✅ Loaded weights: {model_path}")
    print(f"⏳ Scoring {len(pdb_files)} model(s) from {work_dir}...\n")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scorer = ComplexScoringModel(in_node_dim=39, hidden_dim=64).to(device)
    scorer.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    scorer.eval()

    parser = PDBParser(QUIET=True)
    results = []

    for pdb_path in pdb_files:
        pdb_filename = os.path.basename(pdb_path)
        structure = parser.get_structure('hdock_pose', pdb_path)
        model_struct = structure[0]

        try:
            dssp = DSSP(model_struct, pdb_path, dssp='dssp')
        except:
            dssp = {}

        coords, node_features = [], []

        for chain in model_struct:
            for residue in chain:
                res_name = residue.resname.strip().upper()
                if 'CA' in residue and res_name in AA_MAP:
                    atom = residue['CA']
                    is_prot = 1.0
                    id_vec = np.eye(20)[AA_MAP[res_name]].tolist()
                    dssp_key = (chain.id, residue.id)
                    ss = dssp[dssp_key][2] if dssp_key in dssp else '-'
                    sasa = float(dssp[dssp_key][3]) / 200.0 if dssp_key in dssp else 0.0
                    ss_vec = np.eye(8)[SS_MAP.get(ss, 7)].tolist()
                    na_vec = np.zeros(8).tolist()
                elif 'P' in residue and res_name in NA_MAP:
                    atom = residue['P']
                    is_prot = 0.0
                    id_vec = np.zeros(20).tolist()
                    ss_vec = np.zeros(8).tolist()
                    sasa = 1.0
                    na_vec = np.eye(8)[NA_MAP.get(res_name, 0)].tolist()
                else:
                    continue

                coords.append(atom.get_coord())
                feature_vector = [is_prot, 1.0 - is_prot] + id_vec + na_vec + ss_vec + [sasa]
                node_features.append(feature_vector)

        if len(coords) < 10:
            print(f"❌ {pdb_filename}: Error - Could not extract enough valid atoms.")
            continue

        coords = np.array(coords)
        dist_mat = distance_matrix(coords, coords)
        edges = np.where((dist_mat < 10.0) & (dist_mat > 0.1))

        if len(edges[0]) == 0:
            print(f"❌ {pdb_filename}: Error - Graph has no connecting edges.")
            continue

        graph = Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=torch.tensor(np.vstack(edges), dtype=torch.long),
            pos=torch.tensor(coords, dtype=torch.float)
        )

        batch = Batch.from_data_list([graph]).to(device)
        with torch.no_grad():
            raw_logit = scorer(batch).squeeze()
            if raw_logit.dim() == 0:
                raw_logit = raw_logit.unsqueeze(0)
            probability = torch.sigmoid(raw_logit).item()

        results.append((pdb_filename, probability))

    # Sort results by highest probability and print rankings
    results.sort(key=lambda x: x[1], reverse=True)

    print("\n--- FINAL AI RANKINGS ---")
    for rank, (filename, prob) in enumerate(results, 1):
        print(f"Rank {rank}: {filename} -> Native Probability: {prob * 100:.2f}%")

In [ ]:
# @title 🧬 Visualize Rank 2 Complex & Interactions (py3Dmol) { display-mode: "form" }
interaction_cutoff = 4.5 # @param {type:"number"}
image_width = 1200 # @param {type:"integer"}
image_height = 800 # @param {type:"integer"}

!pip install py3Dmol -q

import os
import py3Dmol
from Bio.PDB import PDBParser, NeighborSearch
from IPython.display import display, HTML

if 'results' not in globals() or len(results) < 2:
    print("⚠️ Fewer than 2 ranked models found. Please run the AI Scoring cell first.")
else:
    rank2_filename, rank2_prob = results[1]
    rank2_path = os.path.join(work_dir, rank2_filename)
    print(f"🥈 Rank 2 Model: {rank2_filename} | Native Probability: {rank2_prob * 100:.2f}%")

    # --- 1. Identify Protein–Nucleic Acid Interacting Residues ---
    AA_SET = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
              'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'}
    NA_SET = {'A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC', 'ADE', 'URA', 'GUA', 'CYT', 'THY'}

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('rank2', rank2_path)
    model = structure[0]

    prot_atoms, na_atoms = [], []
    for chain in model:
        for res in chain:
            resname = res.resname.strip().upper()
            if resname in AA_SET:
                prot_atoms.extend(res.get_atoms())
            elif resname in NA_SET:
                na_atoms.extend(res.get_atoms())

    interacting_prot = {}
    interacting_na = {}

    if prot_atoms and na_atoms:
        ns_prot = NeighborSearch(prot_atoms)
        for atom in na_atoms:
            close_prot = ns_prot.search(atom.get_coord(), interaction_cutoff, level='R')
            if close_prot:
                na_res = atom.get_parent()
                na_key = (na_res.get_parent().id, na_res.id[1])
                if na_key not in interacting_na:
                    anchor = na_res['P'].get_coord() if 'P' in na_res else atom.get_coord()
                    interacting_na[na_key] = (na_res.resname.strip(), anchor)
                for p_res in close_prot:
                    p_key = (p_res.get_parent().id, p_res.id[1])
                    if p_key not in interacting_prot:
                        anchor = p_res['CA'].get_coord() if 'CA' in p_res else next(p_res.get_atoms()).get_coord()
                        interacting_prot[p_key] = (p_res.resname.strip(), anchor)

    print(f"🔗 Found {len(interacting_prot)} protein & {len(interacting_na)} nucleic acid residues within {interaction_cutoff} Å.")
    if interacting_prot:
        print("   Protein:", ", ".join(f"{v[0]}{k[1]}({k[0]})" for k, v in sorted(interacting_prot.items())))
    if interacting_na:
        print("   Nucleic Acid:", ", ".join(f"{v[0]}{k[1]}({k[0]})" for k, v in sorted(interacting_na.items())))

    # --- 2. Build py3Dmol Visualization ---
    with open(rank2_path, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=image_width, height=image_height)
    view.addModel(pdb_data, 'pdb')
    view.setBackgroundColor('white')
    view.setStyle({'cartoon': {'colorscheme': 'chain', 'opacity': 0.85}})

    # Highlight & label interacting protein residues (cyan sticks)
    for (chain_id, resi), (resname, coord) in interacting_prot.items():
        view.addStyle({'chain': chain_id, 'resi': resi}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}})
        view.addLabel(
            f"{resname}{resi}:{chain_id}",
            {'position': {'x': float(coord[0]), 'y': float(coord[1]), 'z': float(coord[2])},
             'backgroundColor': '#005f73', 'fontColor': 'white', 'fontSize': 11,
             'backgroundOpacity': 0.85, 'borderThickness': 0}
        )

    # Highlight & label interacting nucleic acid residues (orange sticks)
    for (chain_id, resi), (resname, coord) in interacting_na.items():
        view.addStyle({'chain': chain_id, 'resi': resi}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.2}})
        view.addLabel(
            f"{resname}{resi}:{chain_id}",
            {'position': {'x': float(coord[0]), 'y': float(coord[1]), 'z': float(coord[2])},
             'backgroundColor': '#ae2012', 'fontColor': 'white', 'fontSize': 11,
             'backgroundOpacity': 0.85, 'borderThickness': 0}
        )

    view.zoomTo()

    # --- 3. Display Viewer + High-Res PNG Download Button ---
    png_filename = rank2_filename.replace('.pdb', '_interactions.png')
    download_btn = f"""
    <div style="margin-top: 10px;">
        <button onclick="
            var uri = viewer_{view.uniqueid}.pngURI();
            var a = document.createElement('a');
            a.href = uri;
            a.download = '{png_filename}';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        " style="padding: 10px 18px; font-size: 14px; font-weight: bold; color: white;
                 background-color: #1a73e8; border: none; border-radius: 6px; cursor: pointer;">
            📸 Download High-Res PNG ({image_width}x{image_height})
        </button>
    </div>
    """
    display(HTML(view._make_html() + download_btn))

In [ ]:
# @title 📊 Protein–Nucleic Acid Interaction Table (CSV & High-Res PNG) { display-mode: "form" }
interaction_cutoff = 4.5 # @param {type:"number"}
show_all_atom_pairs = True # @param {type:"boolean"}

import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from Bio.PDB import PDBParser, NeighborSearch

if 'rank2_path' not in globals() or not os.path.exists(rank2_path):
    print("⚠️ Rank 2 PDB not found. Please run the Visualization cell first.")
else:
    AA_SET = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
              'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'}
    NA_SET = {'A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC', 'ADE', 'URA', 'GUA', 'CYT', 'THY'}
    POS_AA = {'ARG', 'LYS', 'HIS'}
    PHOS_ATOMS = {'P', 'OP1', 'OP2', 'O1P', 'O2P'}

    def classify_interaction(p_res, p_atom, na_atom, dist):
        p_el = p_atom.element.upper() if p_atom.element else p_atom.get_name()[0]
        na_el = na_atom.element.upper() if na_atom.element else na_atom.get_name()[0]
        if p_res in POS_AA and p_el == 'N' and na_atom.get_name() in PHOS_ATOMS and dist <= 4.0:
            return "Electrostatic (Salt Bridge)"
        if p_el in {'N', 'O'} and na_el in {'N', 'O'} and dist <= 3.5:
            return "Hydrogen Bond / Polar"
        if p_el == 'C' and na_el == 'C' and dist <= 4.0:
            return "Hydrophobic / π-Stacking"
        return "Van der Waals / Contact"

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('rank2', rank2_path)
    model = structure[0]

    prot_atoms, na_atoms = [], []
    for chain in model:
        for res in chain:
            resname = res.resname.strip().upper()
            if resname in AA_SET:
                prot_atoms.extend(res.get_atoms())
            elif resname in NA_SET:
                na_atoms.extend(res.get_atoms())

    records = []
    if prot_atoms and na_atoms:
        ns_prot = NeighborSearch(prot_atoms)
        for na_atom in na_atoms:
            close_p_atoms = ns_prot.search(na_atom.get_coord(), interaction_cutoff, level='A')
            na_res = na_atom.get_parent()
            na_chain = na_res.get_parent().id
            for p_atom in close_p_atoms:
                p_res = p_atom.get_parent()
                p_chain = p_res.get_parent().id
                dist = float(p_atom - na_atom)
                records.append({
                    'Protein Chain': p_chain,
                    'Protein Residue': f"{p_res.resname.strip()}{p_res.id[1]}",
                    'Protein Atom': p_atom.get_name(),
                    'NA Chain': na_chain,
                    'NA Residue': f"{na_res.resname.strip()}{na_res.id[1]}",
                    'NA Atom': na_atom.get_name(),
                    'Distance (Å)': round(dist, 2),
                    'Interaction Type': classify_interaction(p_res.resname.strip().upper(), p_atom, na_atom, dist)
                })

    if not records:
        print(f"No protein–nucleic acid interactions found within {interaction_cutoff} Å.")
    else:
        df = pd.DataFrame(records).sort_values('Distance (Å)')
        if not show_all_atom_pairs:
            # Keep the closest atom contact per residue-residue pair
            df = df.drop_duplicates(subset=['Protein Chain', 'Protein Residue', 'NA Chain', 'NA Residue'])
        df = df.reset_index(drop=True)

        # --- Save CSV ---
        base_name = rank2_filename.replace('.pdb', '')
        csv_path = os.path.join(work_dir, f"{base_name}_interactions.csv")
        png_path = os.path.join(work_dir, f"{base_name}_interactions_table.png")
        df.to_csv(csv_path, index=False)

        # --- Render High-Resolution PNG Table (300 DPI) ---
        row_height = 0.35
        fig_height = max(2.5, (len(df) + 2) * row_height)
        fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)
        ax.axis('off')
        ax.set_title(
            f"Protein–Nucleic Acid Interactions | {rank2_filename} (Cutoff ≤ {interaction_cutoff} Å)",
            fontsize=13, fontweight='bold', pad=12, color='#1b263b'
        )

        table = ax.table(
            cellText=df.values,
            colLabels=df.columns,
            loc='center',
            cellLoc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)

        for (row, col), cell in table.get_celld().items():
            cell.set_edgecolor('#d0d7de')
            if row == 0:
                cell.set_facecolor('#1b263b')
                cell.set_text_props(color='white', fontweight='bold')
            else:
                cell.set_facecolor('#f6f8fa' if row % 2 == 0 else 'white')

        plt.tight_layout()
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        print(f"✅ Found {len(df)} interaction(s). Saved to Google Drive:\n   • {csv_path}\n   • {png_path}")

        # --- Download Buttons ---
        btn_csv = widgets.Button(description='⬇️ Download CSV', button_style='info')
        btn_png = widgets.Button(description='⬇️ Download High-Res PNG', button_style='success')

        btn_csv.on_click(lambda b: files.download(csv_path))
        btn_png.on_click(lambda b: files.download(png_path))
        display(widgets.HBox([btn_csv, btn_png]))